# MOHIM repeated motif → full-song LoRA

원곡 오디오를 단일 target으로 학습합니다. 기존 ACE-Step 192채널 입력을 그대로 사용하며 `Source 64ch = 반복 motif`, `Mask 64ch = 1`, `Target 64ch = noised full-song`으로 구성합니다.

## 0. Drive와 저장소 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'codex/full-song-motif-source'
REPO_DIR = Path('/content/MOHIM')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
# torchao/flash-attn은 필수가 아니며 Colab torch 조합과 충돌할 수 있어 제외합니다.
def install_filtered(requirements_path, output_path):
    excluded = ('torchao', 'flash-attn')
    lines = requirements_path.read_text(encoding='utf-8').splitlines()
    kept = [line for line in lines if not any(name in line.lower() for name in excluded)]
    output_path.write_text('\n'.join(kept) + '\n', encoding='utf-8')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(output_path)], check=True)

install_filtered(REPO_DIR / 'requirements.txt', Path('/tmp/mohim_full_song_requirements.txt'))
from mohim.trainer import DEFAULT_REVISION, apply_acestep_patch, ensure_acestep_repo
ACESTEP_DIR = ensure_acestep_repo(Path('/content/ACE-Step-1.5'), revision=DEFAULT_REVISION)
apply_acestep_patch(ACESTEP_DIR, REPO_DIR / 'patches/ace-step-1.5-dual-stream.patch')
install_filtered(ACESTEP_DIR / 'requirements.txt', Path('/tmp/mohim_acestep_full_song_requirements.txt'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ACESTEP_DIR), '--no-deps'], check=True)
nano_vllm = ACESTEP_DIR / 'acestep' / 'third_parts' / 'nano-vllm'
if nano_vllm.is_dir():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', str(nano_vllm)], check=True)
print('ACE-Step:', ACESTEP_DIR)

## 1. 분리된 v7 경로와 full-song manifest

In [ ]:
import json
from mohim.manifest import build_full_song_manifest

LORA_VERSION = 'v7_full_song_repeated_motif'
DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset_mean_centered')
MOTIF_VERSION_PATH = DATASET_DIR / '.mohim_motif_version'
MANIFEST_PATH = DATASET_DIR / 'full_song_repeated_motif_manifest.json'
REPEATED_MOTIF_TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM/dual_stream_tensors_repeated_motif')
TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM/full_song_tensors_repeated_motif')
TENSOR_SCHEMA_PATH = TENSOR_DIR / '.mohim_schema'
CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/checkpoints')
LORA_RUN_DIR = Path('/content/drive/MyDrive/MOHIM/lora_run') / LORA_VERSION
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/mohim_lora_checkpoints') / LORA_VERSION
INFERENCE_OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/inference') / LORA_VERSION
MODEL_VARIANT = 'base'
MAX_DURATION = 300.0
DEVICE = 'cuda'
RESET_TENSORS = False
RESET_LORA_RUN = False

assert MOTIF_VERSION_PATH.is_file(), f'motif dataset version이 없습니다: {MOTIF_VERSION_PATH}'
motif_dataset_config = json.loads(MOTIF_VERSION_PATH.read_text(encoding='utf-8'))
allowed_track_ids = motif_dataset_config.get('accepted_track_ids')
assert isinstance(allowed_track_ids, list) and allowed_track_ids, '학습 가능한 track 목록이 없습니다.'
manifest = build_full_song_manifest(DATASET_DIR, MANIFEST_PATH, allowed_track_ids=allowed_track_ids)
assert manifest['metadata']['num_samples'] > 0, 'full-song manifest가 비었습니다.'
assert not manifest['metadata']['skipped'], f'원곡 경로가 없어서 제외된 곡이 있습니다: {manifest["metadata"]["skipped"][:10]}'
print('samples:', manifest['metadata']['num_samples'])
print(json.dumps(manifest['samples'][0], ensure_ascii=False, indent=2))

## 2. 원곡 target 전처리 + 기존 반복 motif 재사용

원곡만 VAE로 새로 인코딩합니다. 이미 만든 v6 반복 motif latent는 다시 인코딩하지 않고 Source 슬롯으로 복사합니다.

In [ ]:
import shutil
import torch

schema = json.dumps({
    'tensor_schema': 'full_song_target_repeated_motif_source_v1',
    'motif_dataset': motif_dataset_config,
}, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
tensor_ready = (
    not RESET_TENSORS
    and TENSOR_SCHEMA_PATH.is_file()
    and TENSOR_SCHEMA_PATH.read_text(encoding='utf-8').strip() == schema
    and next(TENSOR_DIR.glob('*.pt'), None) is not None
)
if RESET_TENSORS and TENSOR_DIR.exists():
    shutil.rmtree(TENSOR_DIR)
TENSOR_DIR.mkdir(parents=True, exist_ok=True)

if not tensor_ready:
    command = [
        sys.executable, 'train.py', 'fixed',
        '--checkpoint-dir', str(CHECKPOINT_DIR),
        '--model-variant', MODEL_VARIANT,
        '--dataset-dir', str(TENSOR_DIR),
        '--output-dir', str(ACESTEP_DIR / 'mohim_full_song_preprocess_run'),
        '--preprocess', '--dataset-json', str(MANIFEST_PATH),
        '--tensor-output', str(TENSOR_DIR), '--max-duration', str(MAX_DURATION),
        '--device', DEVICE, '--precision', 'bf16',
    ]
    subprocess.run(command, cwd=ACESTEP_DIR, check=True)

    repeated_by_motif = {}
    for path in REPEATED_MOTIF_TENSOR_DIR.glob('*.pt'):
        item = torch.load(path, map_location='cpu', weights_only=True)
        motif_path = item.get('metadata', {}).get('motif_seed_audio')
        if motif_path and 'motif_seed_latents' in item:
            repeated_by_motif[str(Path(motif_path).resolve())] = path
    assert repeated_by_motif, f'재사용할 반복 motif tensor가 없습니다: {REPEATED_MOTIF_TENSOR_DIR}'

    converted = 0
    for path in sorted(TENSOR_DIR.glob('*.pt')):
        item = torch.load(path, map_location='cpu', weights_only=True)
        motif_path = item.get('metadata', {}).get('motif_seed_audio')
        repeated_path = repeated_by_motif.get(str(Path(motif_path).resolve())) if motif_path else None
        if repeated_path is None:
            raise KeyError(f'기존 반복 motif를 찾지 못했습니다: {path.name} / {motif_path}')
        repeated_item = torch.load(repeated_path, map_location='cpu', weights_only=True)
        motif_latents = repeated_item['motif_seed_latents']
        target_latents = item['target_latents']
        if motif_latents.shape != target_latents.shape:
            raise ValueError(f'{path.name}: motif/원곡 latent shape 불일치 {tuple(motif_latents.shape)} != {tuple(target_latents.shape)}')
        source_mask = torch.ones_like(motif_latents)
        item['context_latents'] = torch.cat([motif_latents.to(target_latents.dtype), source_mask], dim=-1)
        item['full_song_motif_schema'] = 'source64_repeated_motif_mask64_ones'
        temporary = path.with_name(path.name + '.full-song.tmp')
        torch.save(item, temporary)
        temporary.replace(path)
        converted += 1
    assert converted == manifest['metadata']['num_samples'], f'tensor 수 불일치: {converted}'
    TENSOR_SCHEMA_PATH.write_text(schema + '\n', encoding='utf-8')
    tensor_ready = True

tensor_files = sorted(TENSOR_DIR.glob('*.pt'))
assert tensor_ready and tensor_files
example = torch.load(tensor_files[0], map_location='cpu', weights_only=True)
assert example['context_latents'].shape[-1] == 128
assert example['target_latents'].shape[-1] == 64
assert torch.all(example['context_latents'][..., 64:] == 1)
print('full-song tensors:', len(tensor_files))
print('Source+Mask:', tuple(example['context_latents'].shape), 'Target:', tuple(example['target_latents'].shape))

## 3. 단일-stream LoRA 학습

In [ ]:
import gc
import re

LOCAL_TENSOR_DIR = ACESTEP_DIR / 'mohim_full_song_tensors'
LOCAL_RUN_DIR = ACESTEP_DIR / f'mohim_lora_run_{LORA_VERSION}'
if LOCAL_TENSOR_DIR.exists():
    shutil.rmtree(LOCAL_TENSOR_DIR)
shutil.copytree(TENSOR_DIR, LOCAL_TENSOR_DIR)
if RESET_LORA_RUN:
    for path in (LOCAL_RUN_DIR, LORA_RUN_DIR, DRIVE_CHECKPOINT_DIR):
        if path.exists():
            shutil.rmtree(path)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_epoch(path):
    try:
        return int(torch.load(path / 'training_state.pt', map_location='cpu', weights_only=False).get('epoch', -1))
    except Exception:
        return -1

def valid_resume(path):
    required = ('training_state.pt', 'adapter_model.safetensors', 'run_config.json', 'rng_state.pt')
    return path.is_dir() and all((path / name).is_file() for name in required) and checkpoint_epoch(path) >= 0

resume_candidates = [
    DRIVE_CHECKPOINT_DIR / 'resume_latest',
    DRIVE_CHECKPOINT_DIR / 'resume_latest.previous',
    *DRIVE_CHECKPOINT_DIR.glob('epoch_*'),
]
resume_source = max((path for path in resume_candidates if valid_resume(path)), key=checkpoint_epoch, default=None)
resume_local = None
if resume_source is not None:
    resume_local = LOCAL_RUN_DIR / 'resume_checkpoint'
    if resume_local.exists():
        shutil.rmtree(resume_local)
    shutil.copytree(resume_source, resume_local)
    print('resume:', resume_source)
else:
    print('새 full-song 학습을 epoch 1부터 시작합니다.')

In [ ]:
os.environ['MOHIM_CHECKPOINT_BACKUP_DIR'] = str(DRIVE_CHECKPOINT_DIR)
os.environ['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, '-u', 'train.py', '--plain', '--yes', 'fixed',
    '--checkpoint-dir', str(CHECKPOINT_DIR), '--model-variant', MODEL_VARIANT,
    '--dataset-dir', str(LOCAL_TENSOR_DIR), '--output-dir', str(LOCAL_RUN_DIR),
    '--rank', '64', '--alpha', '128', '--dropout', '0.1',
    '--batch-size', '1', '--gradient-accumulation', '4',
    '--epochs', '100', '--save-every', '10', '--lr', '0.0003', '--val-split', '0.2',
    '--shift', '1.0', '--num-inference-steps', '50',
    '--optimizer-type', 'adamw8bit', '--scheduler-type', 'cosine_restarts',
    '--warmup-steps', '100', '--weight-decay', '0.01', '--max-grad-norm', '1.0',
    '--seed', '42', '--num-workers', '4', '--log-every', '10',
    '--device', DEVICE, '--precision', 'bf16',
]
if resume_local is not None:
    command.extend(['--resume-from', str(resume_local)])
subprocess.run(command, cwd=ACESTEP_DIR, check=True)
FINAL_ADAPTER_DIR = LOCAL_RUN_DIR / 'final'
assert (FINAL_ADAPTER_DIR / 'adapter_model.safetensors').is_file()
shutil.copytree(LOCAL_RUN_DIR, LORA_RUN_DIR, dirs_exist_ok=True)
print('saved:', LORA_RUN_DIR)

## 4. Full-song inference

In [ ]:
import soundfile as sf
from peft import PeftModel
from acestep.training.dataset_builder_modules.preprocess_encoder import run_encoder
from acestep.training.dataset_builder_modules.preprocess_lyrics import encode_lyrics
from acestep.training.dataset_builder_modules.preprocess_text import encode_text
from acestep.training_v2.dual_stream_preprocess import _encode_audio, encode_motif_condition_latents
from acestep.training_v2.model_loader import load_decoder_for_training, load_text_encoder, load_vae, unload_models
from acestep.training_v2.preprocess_prompt import build_simple_prompt

CUSTOM_MOTIF_PATH = Path('/content/motif.wav')
CUSTOM_MOTIF_STEM = 'piano'
CUSTOM_MOTIF_START_SECONDS = 0.0
CUSTOM_LANGUAGE = 'English'
CUSTOM_LYRICS = '''
Paste the complete lyrics here.
'''
CUSTOM_OUTPUT_NAME = 'custom_full_song'
INFERENCE_DURATION_SECONDS = 180.0  # 원하는 전체 곡 길이로 수정, 최대 300초 권장
INFERENCE_STEPS = 50
INFERENCE_SEED = 42
CFG_SCALE = 7.0
EXPERIMENTS = ('motif',)  # 비교가 필요하면 ('motif', 'no_motif')
CHECKPOINT_OVERRIDE = None

assert CUSTOM_MOTIF_PATH.is_file()
assert CUSTOM_LYRICS.strip() and 'Paste the complete lyrics' not in CUSTOM_LYRICS
checkpoint_pattern = re.compile(r'^epoch_(\d+)_loss_([0-9.]+)$')
checkpoint_candidates = []
for path in DRIVE_CHECKPOINT_DIR.glob('epoch_*_loss_*'):
    match = checkpoint_pattern.match(path.name)
    if match and (path / 'adapter_model.safetensors').is_file():
        checkpoint_candidates.append((float(match.group(2)), int(match.group(1)), path))
for path in (DRIVE_CHECKPOINT_DIR / 'resume_latest', DRIVE_CHECKPOINT_DIR / 'resume_latest.previous'):
    config_path = path / 'run_config.json'
    if (path / 'adapter_model.safetensors').is_file() and config_path.is_file():
        loss = json.loads(config_path.read_text(encoding='utf-8')).get('val_loss')
        epoch = checkpoint_epoch(path)
        if loss is not None and epoch >= 0:
            checkpoint_candidates.append((float(loss), epoch, path))
if CHECKPOINT_OVERRIDE is not None:
    BEST_ADAPTER_DIR = Path(CHECKPOINT_OVERRIDE)
    BEST_LOSS, BEST_EPOCH = None, checkpoint_epoch(BEST_ADAPTER_DIR)
else:
    assert checkpoint_candidates, f'checkpoint가 없습니다: {DRIVE_CHECKPOINT_DIR}'
    BEST_LOSS, BEST_EPOCH, BEST_ADAPTER_DIR = min(checkpoint_candidates, key=lambda row: row[0])
print('checkpoint:', BEST_ADAPTER_DIR, 'epoch:', BEST_EPOCH, 'loss:', BEST_LOSS)

In [ ]:
dtype = torch.bfloat16
generation_duration = INFERENCE_DURATION_SECONDS
target_samples = round(generation_duration * 48000)
motif_duration = sf.info(str(CUSTOM_MOTIF_PATH)).duration
vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
motif_source = encode_motif_condition_latents(
    str(CUSTOM_MOTIF_PATH), vae, dtype, target_samples=target_samples,
    motif_start_sec=CUSTOM_MOTIF_START_SECONDS,
    motif_end_sec=CUSTOM_MOTIF_START_SECONDS + motif_duration,
)
silence_source = _encode_audio(torch.zeros(2, target_samples), vae, dtype) if 'no_motif' in EXPERIMENTS else None
unload_models(vae)
del vae
assert silence_source is None or motif_source.shape == silence_source.shape

caption = (f'Full {CUSTOM_LANGUAGE.strip()} pop song with vocals and complete instrumentation, ' f'organized around a recurring {CUSTOM_MOTIF_STEM.strip().lower()} motif.')
text_tokenizer, text_encoder = load_text_encoder(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
prompt = build_simple_prompt({'caption': caption, 'duration': generation_duration, 'bpm': None, 'timesignature': '', 'keyscale': ''})
text_hs, text_mask = encode_text(text_encoder, text_tokenizer, prompt, DEVICE, dtype)
lyric_hs, lyric_mask = encode_lyrics(text_encoder, text_tokenizer, CUSTOM_LYRICS.strip(), DEVICE, dtype)
unload_models(text_encoder)
del text_encoder, text_tokenizer

model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
encoder_hs, encoder_mask = run_encoder(model, text_hs, text_mask, lyric_hs, lyric_mask, DEVICE, dtype)
del text_hs, text_mask, lyric_hs, lyric_mask
model.decoder = PeftModel.from_pretrained(model.decoder, str(BEST_ADAPTER_DIR), is_trainable=False).to(device=DEVICE, dtype=dtype).eval()

@torch.no_grad()
def sample_full_song(source_latents):
    generator = torch.Generator(device=DEVICE).manual_seed(INFERENCE_SEED)
    latents = torch.randn((1, *source_latents.shape), device=DEVICE, dtype=dtype, generator=generator)
    source = source_latents.unsqueeze(0).to(DEVICE, dtype=dtype)
    source_mask = torch.ones_like(source)
    context = torch.cat([source, source_mask], dim=-1)
    attention_mask = torch.ones(1, source.shape[1], device=DEVICE, dtype=dtype)
    condition = encoder_hs.to(DEVICE, dtype=dtype)
    condition_mask = encoder_mask.to(DEVICE, dtype=dtype)
    unconditional = model.null_condition_emb.to(DEVICE, dtype=dtype).expand_as(condition)
    times = torch.linspace(1.0, 0.0, INFERENCE_STEPS + 1, device=DEVICE, dtype=dtype)
    for index in range(INFERENCE_STEPS):
        timestep = times[index].expand(1)
        combined = model.decoder(
            hidden_states=torch.cat([latents, latents]),
            timestep=torch.cat([timestep, timestep]), timestep_r=torch.cat([timestep, timestep]),
            attention_mask=torch.cat([attention_mask, attention_mask]),
            encoder_hidden_states=torch.cat([condition, unconditional]),
            encoder_attention_mask=torch.cat([condition_mask, condition_mask]),
            context_latents=torch.cat([context, context]),
        )[0]
        conditional_velocity, unconditional_velocity = combined.chunk(2)
        velocity = unconditional_velocity + CFG_SCALE * (conditional_velocity - unconditional_velocity)
        latents = latents + (times[index + 1] - times[index]) * velocity
    return latents.cpu()

generated_latents = {}
for experiment in EXPERIMENTS:
    source = motif_source if experiment == 'motif' else silence_source
    generated_latents[experiment] = sample_full_song(source)
    print(experiment, 'complete')
del model, motif_source, silence_source, encoder_hs, encoder_mask
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import math
import torch.nn.functional as F
from IPython.display import Audio, display

def decode_latents_tiled(vae, latents_btc, chunk_frames=256, overlap=64):
    latents = latents_btc.transpose(1, 2)
    stride = chunk_frames - 2 * overlap
    decoded, factor = [], None
    for index in range(math.ceil(latents.shape[-1] / stride)):
        core_start = index * stride
        core_end = min(core_start + stride, latents.shape[-1])
        window_start = max(0, core_start - overlap)
        window_end = min(latents.shape[-1], core_end + overlap)
        chunk = latents[:, :, window_start:window_end].to(DEVICE, dtype=vae.dtype)
        with torch.inference_mode():
            audio = vae.decode(chunk).sample
        factor = factor or audio.shape[-1] / chunk.shape[-1]
        trim_start = round((core_start - window_start) * factor)
        trim_end = round((window_end - core_end) * factor)
        end = audio.shape[-1] - trim_end if trim_end else audio.shape[-1]
        decoded.append(audio[:, :, trim_start:end].float().cpu())
    return torch.cat(decoded, dim=-1)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
outputs = {}
for experiment, latents in generated_latents.items():
    audio = decode_latents_tiled(vae, latents)
    if audio.shape[-1] < target_samples:
        audio = F.pad(audio, (0, target_samples - audio.shape[-1]))
    audio = audio[:, :, :target_samples]
    audio = audio / audio.abs().amax().clamp_min(1.0)
    output_dir = INFERENCE_OUTPUT_DIR / BEST_ADAPTER_DIR.name / CUSTOM_OUTPUT_NAME / experiment
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'generated_full_song.wav'
    sf.write(path, audio.squeeze(0).transpose(0, 1).numpy(), 48000)
    outputs[experiment] = path
    print(experiment, path)
    display(Audio(filename=str(path)))
unload_models(vae)
del vae, generated_latents
gc.collect()
torch.cuda.empty_cache()

## 5. Generated full song ↔ motif 유사도

In [ ]:
import numpy as np
import pandas as pd
import librosa
from sklearn.metrics.pairwise import cosine_similarity
from acestep.training.dataset_builder_modules.preprocess_audio import load_audio_stereo
from acestep.training_v2.dual_stream_preprocess import build_repeated_motif_canvas
from mohim.motif import _max_shifted_cosine_similarity, _resize_time

def features(audio):
    hop, n_fft = 512, 2048
    preroll = n_fft // hop
    padded = np.pad(audio, (preroll * hop, 0))
    padded_onset = librosa.onset.onset_strength(y=padded, sr=48000, hop_length=hop, n_fft=n_fft)
    rms = librosa.feature.rms(y=audio, hop_length=hop)[0]
    onset = _resize_time(padded_onset[preroll:preroll + len(rms)][None], 256).ravel()
    chroma = _resize_time(librosa.feature.chroma_cens(y=audio, sr=48000, hop_length=hop), 64)
    centered_chroma = chroma - chroma.mean(axis=0, keepdims=True)
    return onset, chroma.ravel(), centered_chroma.ravel()

motif_audio, _ = load_audio_stereo(str(CUSTOM_MOTIF_PATH), 48000, generation_duration)
reference = build_repeated_motif_canvas(
    motif_audio, target_samples, motif_start_sec=CUSTOM_MOTIF_START_SECONDS,
    motif_end_sec=CUSTOM_MOTIF_START_SECONDS + motif_duration, sample_rate=48000,
).mean(0).numpy()
reference_onset, reference_chroma, reference_centered_chroma = features(reference)
rows = []
for experiment, path in outputs.items():
    generated, _ = librosa.load(str(path), sr=48000, mono=True)
    onset, chroma, centered_chroma = features(generated)
    raw_onset = _max_shifted_cosine_similarity(reference_onset, onset, 3, cosine_similarity, mean_center=False)
    centered_onset = _max_shifted_cosine_similarity(reference_onset, onset, 3, cosine_similarity, mean_center=True)
    raw_chroma = float(cosine_similarity(reference_chroma[None], chroma[None])[0, 0])
    centered_chroma_score = float(cosine_similarity(reference_centered_chroma[None], centered_chroma[None])[0, 0])
    rows.append({
        'experiment': experiment, 'onset_similarity': raw_onset, 'chroma_similarity': raw_chroma,
        'mean_centered_onset_similarity': centered_onset,
        'mean_centered_chroma_similarity': centered_chroma_score,
        'similarity': 0.7 * centered_onset + 0.3 * centered_chroma_score,
    })
table = pd.DataFrame(rows)
display(table.style.format({column: '{:.4f}' for column in table.columns if column != 'experiment'}))
score_dir = INFERENCE_OUTPUT_DIR / BEST_ADAPTER_DIR.name / CUSTOM_OUTPUT_NAME
table.to_csv(score_dir / 'full_song_motif_similarity.csv', index=False)
(score_dir / 'full_song_motif_similarity.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')